In [1]:
import sys
sys.path.append('../src')

from datetime import date, datetime, timedelta

from bank.bank import Bank
from bank.client import Client, Contact

from account.bank_account import BankAccount
from account.savings_account import SavingsAccount
from account.premium_account import PremiumAccount
from account.investment_account import InvestmentAccount, Asset

from account.enums import AccountCurrency
from account.enums import AssetType

from transaction.transaction import Transaction
from transaction.transaction_queue import TransactionAlreadyInQueue, QueueIsEmpty

bank = Bank()
print("Bank created")

clients = [
    Client(
        name="Иван Иванов",
        client_id="c1",
        date_of_birth=date(1993, 2, 15),
        login="ivan",
        password="1234",
        contacts=[Contact("email", "ivan@mail.com")],
    ),
    Client(
        name="Мария Петрова",
        client_id="c2",
        date_of_birth=date(1998, 7, 20),
        login="maria",
        password="abcd",
        contacts=[Contact("phone", "+79991234567")]
    ),
    Client(
        name="Сергей Сидоров",
        client_id="c3",
        date_of_birth=date(1983, 11, 5),
        login="sergey",
        password="qwerty",
        contacts=[Contact("email", "sergey@mail.com")]
    ),
]

for c in clients:
    bank.add_client(c)

print("Clients:", [c.name for c in bank.clients])

ivan, maria, sergey = clients

ivan_acc1 = BankAccount(owner_name=ivan.name)
ivan_acc2 = SavingsAccount(owner_name=ivan.name, min_balance=100, monthly_interest=1)

maria_acc1 = PremiumAccount(owner_name=maria.name, fee=5, overdraft_limit=500)

sergey_acc1 = BankAccount(owner_name=sergey.name)

bank.open_account(ivan.client_id, ivan_acc1)
bank.open_account(ivan.client_id, ivan_acc2)

bank.open_account(maria.client_id, maria_acc1)

bank.open_account(sergey.client_id, sergey_acc1)

print("Accounts created")

ivan_acc1.deposit(20000)
ivan_acc2.deposit(5000)

maria_acc1.deposit(10000)

sergey_acc1.deposit(1000)

print("Balances:")
print("Ivan:", ivan_acc1.balance, ivan_acc2.balance)
print("Maria:", maria_acc1.balance)
print("Sergey:", sergey_acc1.balance)

queue = bank.transaction_queue

t1 = Transaction(1000, AccountCurrency.USD, 1, ivan_acc1, maria_acc1)
t2 = Transaction(2000, AccountCurrency.USD, 1, ivan_acc1, sergey_acc1)
t3 = Transaction(300, AccountCurrency.USD, 1, maria_acc1, ivan_acc1)
t4 = Transaction(1500, AccountCurrency.USD, 1, ivan_acc2, maria_acc1)

# отложенная транзакция
t5 = Transaction(
    500,
    AccountCurrency.USD,
    1,
    maria_acc1,
    sergey_acc1,
    execute_time=datetime.now() + timedelta(seconds=10)
)

t6 = Transaction(700, AccountCurrency.USD, 1, ivan_acc1, maria_acc1)
t7 = Transaction(400, AccountCurrency.USD, 1, maria_acc1, ivan_acc2)
t8 = Transaction(600, AccountCurrency.USD, 1, ivan_acc2, sergey_acc1)

# транзакция на замороженный счет
t9 = Transaction(200, AccountCurrency.USD, 1, ivan_acc1, sergey_acc1)

# обычная
t10 = Transaction(900, AccountCurrency.USD, 1, maria_acc1, ivan_acc1)

transactions = [t1,t2,t3,t4,t5,t6,t7,t8,t9,t10]

for t in transactions:
    queue.enqueue(t)

print("10 transactions added")

try:
    queue.enqueue(t1)
except TransactionAlreadyInQueue:
    print("Duplicate transaction detected")

bank.freeze_account(sergey_acc1.id)

print("Sergey account frozen")

queue.cancel(t8.transaction_id)

print("Transaction t8 cancelled")

processor = bank.transaction_processor

while True:
    try:
        processor.process()
    except QueueIsEmpty:
        print("Queue is empty")
        break

print("Final balances")

print("Ivan acc1:", ivan_acc1.balance)
print("Ivan acc2:", ivan_acc2.balance)

print("Maria:", maria_acc1.balance)

print("Sergey:", sergey_acc1.balance)

for t in transactions:
    reject = None

    if t.reject_reason:
        reject = f"{type(t.reject_reason).__name__}: {t.reject_reason}"

    print(
        t.transaction_id,
        t.status.name,
        "reject:",
        reject
    )

Bank created
Clients: ['Иван Иванов', 'Мария Петрова', 'Сергей Сидоров']
Accounts created
Balances:
Ivan: 20000 5000
Maria: 10000
Sergey: 1000
10 transactions added
Duplicate transaction detected
Sergey account frozen
Transaction t8 cancelled
Queue is empty
Final balances
Ivan acc1: 19488
Ivan acc2: 3896
Maria: 11553
Sergey: 1000
2a758812-482f-4054-bcbe-9d869066ce8f EXECUTED reject: None
e217a65a-fcb6-4b8e-98ff-79051b74f154 REJECTED reject: AccountFrozenError: 
7cc6689e-ff1b-4a39-a71d-7116db9efb84 EXECUTED reject: None
f0ae3e55-960c-4972-bf86-5b8f64335d75 EXECUTED reject: None
7cbb6177-26a0-4e37-8d18-362392f8153d CREATED reject: None
ebe0cfad-ec56-4e3b-a008-53ae07c3135b EXECUTED reject: None
a152feda-1bb4-4477-9e22-70143937c715 EXECUTED reject: None
c09357b9-3d34-4cd8-9ab9-47da1620e740 CANCELED reject: None
c51db6e3-dd21-438c-a093-d13927606cfd REJECTED reject: AccountFrozenError: 
06b28a71-8f87-465b-bbfc-1f07a50962cb EXECUTED reject: None
  client_id            name  status date_of_bir